<a href="https://colab.research.google.com/github/swarnkarnitin/TrafficMonitoring/blob/main/Traffic_Analysis_Optimize_FPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install ultralytics opencv-python

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Following cell has the final code

In [ ]:
import cv2
from ultralytics import YOLO
import os

def count_objects_in_segment(video_path, model, start_frame, end_frame, frame_skip_interval=1):
    """
    Counts objects in a specified video segment with a given frame skip interval.

    Args:
        video_path (str): Path to the video file.
        model (YOLO): Loaded YOLO model.
        start_frame (int): The starting frame of the segment (inclusive).
        end_frame (int): The ending frame of the segment (exclusive).
        frame_skip_interval (int): The number of frames to skip between processing.

    Returns:
        dict: A dictionary containing the total counts of each detected object class
              within the processed frames of the segment.
    """
    video_cap = cv2.VideoCapture(video_path)
    if not video_cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return None

    video_cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    current_counts = {}
    processed_frames_count = 0

    for i in range(start_frame, end_frame, frame_skip_interval):
        video_cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = video_cap.read()
        if not ret:
            # If we can't read the frame we are supposed to process after seeking,
            # it might mean we are at the end or there's an issue.
            # However, the loop range should handle the end correctly.
            # This break is mainly for unexpected read errors within the range.
            break

        processed_frames_count += 1

        # Perform object detection
        results = model(frame)

        # Process the detection results and update counts
        for r in results:
            boxes = r.boxes
            for box in boxes:
                class_id = int(box.cls[0])
                label = model.names[class_id]
                if label not in current_counts:
                    current_counts[label] = 0
                current_counts[label] += 1

    video_cap.release()
    return current_counts

# --- Main Analysis ---

# Define the path to your video file
video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV" # Change this to your video path

# Load a pre-trained YOLOv8 model
# Make sure you have run the cell to install ultralytics and opencv-python
model = YOLO("yolov8n.pt")

# Check if the video file exists
if not os.path.exists(video_path):
    print(f"Error: Video file not found at {video_path}")
else:
    # Get video properties to define the segment
    video_cap_check = cv2.VideoCapture(video_path)
    if not video_cap_check.isOpened():
         print(f"Error: Could not open video file {video_path} for initial check.")
    else:
        total_frames = int(video_cap_check.get(cv2.CAP_PROP_FRAME_COUNT))
        video_cap_check.release() # Release the capture object after getting properties

        # Define the start and end frame numbers for a representative segment
        start_frame = 0
        end_frame = min(200, total_frames) # Process the first 200 frames or fewer

        print(f"Analyzing segment from frame {start_frame} to {end_frame-1}")

        # 1. Initial Object Counting (Baseline)
        print("\n--- Establishing Baseline Counts (Frame Skip = 1) ---")
        baseline_counts = count_objects_in_segment(video_path, model, start_frame, end_frame, frame_skip_interval=1)

        if baseline_counts is not None:
            print("Baseline object counts:")
            print(baseline_counts)

            # 2. Iterative Frame Skipping
            print("\n--- Starting Iterative Frame Skipping Analysis ---")
            frame_skip_interval = 2 # Start with skipping 1 frame (process every 2nd frame)
            maximum_safe_frame_skip = 1 # Initialize maximum safe skip to 1 (processing every frame)

            # Define a tolerance for count comparison (e.g., 5% difference)
            count_tolerance_percentage = 5

            while frame_skip_interval < (end_frame - start_frame):
                print(f"\nTesting frame skip interval: {frame_skip_interval}")
                current_counts = count_objects_in_segment(video_path, model, start_frame, end_frame, frame_skip_interval)

                if current_counts is None:
                    print("Error during object counting with frame skipping. Stopping analysis.")
                    break

                print(f"Object counts at skip interval {frame_skip_interval}: {current_counts}")

                # Compare current counts with baseline counts
                counts_changed_significantly = False
                all_classes = set(baseline_counts.keys()).union(set(current_counts.keys()))

                for label in all_classes:
                    baseline_count = baseline_counts.get(label, 0)
                    current_count = current_counts.get(label, 0)

                    # Avoid division by zero if baseline count is 0
                    if baseline_count > 0:
                        percentage_diff = abs(current_count - baseline_count) / baseline_count * 100
                        if percentage_diff > count_tolerance_percentage:
                            counts_changed_significantly = True
                            print(f"  Significant change for class '{label}': Baseline={baseline_count}, Current={current_count}, Difference={percentage_diff:.2f}%")
                            break # No need to check other classes if one changed significantly
                    elif current_count > 0:
                         # If baseline is 0 but current is > 0, it's a significant change
                         counts_changed_significantly = True
                         print(f"  Significant change for class '{label}': Baseline={baseline_count}, Current={current_count}")
                         break


                if counts_changed_significantly:
                    maximum_safe_frame_skip = frame_skip_interval - 1
                    print(f"\nCounts changed significantly at frame skip interval {frame_skip_interval}.")
                    print(f"Maximum safe frame skip interval: {maximum_safe_frame_skip}")
                    break # Stop when counts change significantly
                else:
                    print("  Counts are within tolerance. Increasing frame skip interval.")
                    maximum_safe_frame_skip = frame_skip_interval # Update maximum safe skip
                    frame_skip_interval += 1

            # If the loop finishes without significant change, the last tested skip interval is the maximum safe one
            if not counts_changed_significantly:
                 print("\nFinished testing all skip intervals up to the segment length.")
                 print(f"Maximum safe frame skip interval for this segment: {maximum_safe_frame_skip}")

            # 3. Report Maximum Safe Frame Skip
            print("\n--- Analysis Complete ---")
            print(f"Determined maximum safe frame skip interval: {maximum_safe_frame_skip}")

        else:
            print("\n--- Analysis Failed ---")
            print("Could not establish baseline counts. Please check the video path and file integrity.")

In [ ]:
# import cv2
# import os
# from ultralytics import YOLO

# # Define output video file path and filename
# output_video_path = "output_video.avi"

# # Re-open video file to ensure it's at the beginning
# video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV"
# video_cap = cv2.VideoCapture(video_path)

# # Check if the video capture was successful
# if not video_cap.isOpened():
#     print(f"Error: Could not open video file {video_path}")
# else:
#     # Get video properties from the re-opened video_cap
#     frame_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     frame_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#     fps = int(video_cap.get(cv2.CAP_PROP_FPS))

#     print(f"Frame Width: {frame_width}, Frame Height: {frame_height}, FPS: {fps}")

#     # Define the video codec
#     fourcc = cv2.VideoWriter_fourcc(*'XVID')

#     # Create VideoWriter object
#     out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

#     print(f"VideoWriter initialized: {out.isOpened()}")

#     if not out.isOpened():
#         print(f"Error: Could not initialize VideoWriter with codec {fourcc} for path {output_video_path}")
#     else:
#         # Load the best.pt model
#         model = YOLO("/content/drive/MyDrive/Traffic_Videos/Trained_Model/best.pt")

#         # Now start the processing loop
#         while True:
#             ret, frame = video_cap.read()
#             if not ret:
#                 break

#             # Run YOLO detection
#             results = model(frame)

#             # Draw bounding boxes for detected objects (using the class names from your trained model)
#             for r in results:
#                 boxes = r.boxes
#                 for box in boxes:
#                     x1, y1, x2, y2 = box.xyxy[0]
#                     confidence = box.conf[0]
#                     class_id = box.cls[0]
#                     label = model.names[int(class_id)]
#                     # You can add a confidence threshold here if needed
#                     # if confidence > 0.5:
#                     cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
#                     cv2.putText(frame, f"{label} {confidence:.2f}", (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

#             # Write the processed frame to the output video file
#             out.write(frame)

#         print("Video processing finished.")

#     # Release video capture and writer objects
#     video_cap.release()
#     if out.isOpened():
#         out.release()

# cv2.destroyAllWindows()

Streaming output truncated to the last 5000 lines.
Speed: 2.4ms preprocess, 890.7ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 13 cars, 6 motorbikes, 2 Autos, 909.4ms
Speed: 2.3ms preprocess, 909.4ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 13 cars, 8 motorbikes, 2 Autos, 894.3ms
Speed: 2.2ms preprocess, 894.3ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 13 cars, 7 motorbikes, 2 Autos, 902.5ms
Speed: 3.6ms preprocess, 902.5ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 13 cars, 7 motorbikes, 1 Auto, 918.8ms
Speed: 2.1ms preprocess, 918.8ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 13 cars, 8 motorbikes, 1 Auto, 982.5ms
Speed: 2.5ms preprocess, 982.5ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 13 cars, 7 motorbikes, 1 Auto, 1144.2ms
Speed: 2.5ms preprocess, 1144.2ms inference, 1.5ms 

In [ ]:
import os
import zipfile

zip_file_path = '/content/drive/My Drive/Traffic_Videos/roboflow_images/yolo_final_dataset.zip'
extract_dir = './yolo_final_dataset'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Unzip the file
if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Successfully unzipped {zip_file_path} to {extract_dir}")
else:
    print(f"Error: {zip_file_path} not found.")

Successfully unzipped /content/drive/My Drive/Traffic_Videos/roboflow_images/yolo_final_dataset.zip to ./yolo_final_dataset


Training the model on Roboflow data

In [ ]:
# from ultralytics import YOLO

# # Load a YOLOv8n model
# model = YOLO("yolo11m.pt")  # You can choose a different model size like yolov8s.pt, yolov8m.pt, etc.

# # Train the model on the custom dataset
# # Assuming your dataset has a data.yaml file in the extracted directory
# data_config_path = '/content/yolo_final_dataset/yolo_final_dataset/data.yaml'

# if os.path.exists(data_config_path):
#     results = model.train(data=data_config_path, epochs=50, imgsz=640) # You can adjust epochs and imgsz
#     print("Training finished.")
# else:
#     print(f"Error: data.yaml not found at {data_config_path}. Please ensure your dataset is correctly extracted and contains a data.yaml file.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.179 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_final_dataset/yolo_final_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou

Overriding model.yaml nc=80 with nc=11

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralytics.nn.modules.block.C3k2            [128, 256, 1, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  1    444928  ultralytics.nn.modules.block.C3k2            [256, 512, 1, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512, 512, 1, True]           


  7                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  8                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512, 512, 1, True]           
  9                  -1  1    656896  ultralytics.nn.modules.block.SPPF            [512, 512, 5]                 
 10                  -1  1    990976  ultralytics.nn.modules.block.C2PSA           [512, 512, 1]                 
 11                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 12             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 13                  -1  1   1642496  ultralytics.nn.modules.block.C3k2            [1024, 512, 1, True]          
 14                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 15             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]  

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1463.5±716.3 MB/s, size: 63.3 KB)


train: Scanning /content/yolo_final_dataset/yolo_final_dataset/train/labels... 7314 images, 1 backgrounds, 0 corrupt: 100%|██████████| 7315/7315 [00:03<00:00, 2066.48it/s]


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      37/50      8.29G      1.332     0.8263       1.25         57        640: 100%|██████████| 458/458 [04:23<00:00,  1.74it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 33/33 [00:16<00:00,  2.04it/s]

                   all       1039       5209      0.571      0.635      0.668      0.421



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      38/50      8.19G      1.333     0.8214      1.257        121        640:  79%|███████▉  | 361/458 [03:27<00:54,  1.78it/s]

Running object detection on our data

In [ ]:
# import cv2

# # Define output video file path and filename
# output_video_path = "output_video.avi"

# # Re-open video file to ensure it's at the beginning
# video_cap = cv2.VideoCapture("/content/drive/My Drive/Traffic_Videos/Mundhawan_Circlet2.MOV")

# # Get video properties from the re-opened video_cap
# frame_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# frame_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# fps = int(video_cap.get(cv2.CAP_PROP_FPS))

# print(f"Frame Width: {frame_width}, Frame Height: {frame_height}, FPS: {fps}")

# # Define the video codec
# fourcc = cv2.VideoWriter_fourcc(*'XVID')

# # Create VideoWriter object
# out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# print(f"VideoWriter initialized: {out.isOpened()}")

# # Now start the processing loop
# while True:
#   ret, frame = video_cap.read()
#   if not ret:
#     break

#   # Run YOLO detection
#   results = model(frame)

#   # Draw bounding boxes for detected cars
#   for result in results[0].boxes.data.tolist():
#     x1, y1, x2, y2, confidence, class_id = result
#     if int(class_id) == 2: # Class ID 2 corresponds to 'car' in COCO dataset
#       cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)

#   # Write the processed frame to the output video file
#   out.write(frame)

# # Release video capture and writer objects
# video_cap.release()
# out.release()
# cv2.destroyAllWindows()

In [ ]:
# # Open video file
# video_cap = cv2.VideoCapture("/content/drive/My Drive/Traffic_Videos/Mundhawan_Circlet2.MOV")

# while True:
#   ret, frame = video_cap.read()
#   if not ret:
#     break

#   # Run YOLO detection
#   results = model(frame)

#   # Draw bounding boxes for detected cars
#   for result in results[0].boxes.data.tolist():
#     x1, y1, x2, y2, confidence, class_id = result
#     if int(class_id) == 2: # Class ID 2 corresponds to 'car' in COCO dataset
#       cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)

#   cv2.imshow("Car Detection", frame)
#   if cv2.waitKey(1) & 0xFF == ord('q'):
#     break

# video_cap.release()
# cv2.destroyAllWindows()

In [ ]:
# video_cap = cv2.VideoCapture("/content/drive/My Drive/Traffic_Videos/Mundhawan_Circlet2.MOV")

**Reasoning**:
Call the loaded YOLO model on the current frame and store the results.



In [ ]:
# for result in results[0].boxes.data.tolist():
#   x1, y1, x2, y2, confidence, class_id = result
#   if int(class_id) == 2: # Class ID 2 corresponds to 'car' in COCO dataset
#     cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)

In [ ]:
# import cv2
# import os

# # Define output video file path and filename
# output_video_path = "output_video.avi"

# # Re-open video file to ensure it's at the beginning
# video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV"
# video_cap = cv2.VideoCapture(video_path)

# # Check if the video capture was successful
# if not video_cap.isOpened():
#     print(f"Error: Could not open video file {video_path}")
# else:
#     # Get video properties from the re-opened video_cap
#     frame_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     frame_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#     fps = int(video_cap.get(cv2.CAP_PROP_FPS))

#     print(f"Frame Width: {frame_width}, Frame Height: {frame_height}, FPS: {fps}")

#     # Define the video codec
#     fourcc = cv2.VideoWriter_fourcc(*'XVID')

#     # Create VideoWriter object
#     out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

#     print(f"VideoWriter initialized: {out.isOpened()}")

#     if not out.isOpened():
#         print(f"Error: Could not initialize VideoWriter with codec {fourcc} for path {output_video_path}")
#     else:
#         # Load the best.pt model
#         model = YOLO("./runs/detect/train/weights/best.pt")

#         # Now start the processing loop
#         while True:
#             ret, frame = video_cap.read()
#             if not ret:
#                 break

#             # Run YOLO detection
#             results = model(frame)

#             # Draw bounding boxes for detected objects (using the class names from your trained model)
#             for r in results:
#                 boxes = r.boxes
#                 for box in boxes:
#                     x1, y1, x2, y2 = box.xyxy[0]
#                     confidence = box.conf[0]
#                     class_id = box.cls[0]
#                     label = model.names[int(class_id)]
#                     # You can add a confidence threshold here if needed
#                     # if confidence > 0.5:
#                     cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
#                     cv2.putText(frame, f"{label} {confidence:.2f}", (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

#             # Write the processed frame to the output video file
#             out.write(frame)

#         print("Video processing finished.")

#     # Release video capture and writer objects
#     video_cap.release()
#     if out.isOpened():
#         out.release()

# cv2.destroyAllWindows()

Streaming output truncated to the last 5000 lines.
Speed: 2.0ms preprocess, 8.0ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 24 cars, 1 microbus, 6 motorbikes, 1 pickup-van, 8.0ms
Speed: 1.8ms preprocess, 8.0ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 25 cars, 1 microbus, 7 motorbikes, 1 pickup-van, 12.4ms
Speed: 1.9ms preprocess, 12.4ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 23 cars, 9 motorbikes, 1 pickup-van, 8.2ms
Speed: 1.8ms preprocess, 8.2ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 21 cars, 8 motorbikes, 1 pickup-van, 11.3ms
Speed: 1.9ms preprocess, 11.3ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 22 cars, 8 motorbikes, 1 pickup-van, 8.1ms
Speed: 1.7ms preprocess, 8.1ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 22 cars, 9 motorbikes, 1 pickup-van, 9.7ms
Speed: 1.9ms pr

In [ ]:
# import os
# import shutil

# source_path = "output_video.avi"
# destination_path = "/content/output_video.avi"

# if os.path.exists(source_path):
#     shutil.copy(source_path, destination_path)
#     print(f"Successfully copied {source_path} to {destination_path}")
# else:
#     print(f"Error: {source_path} not found.")

Error: output_video.avi not found.


In [ ]:
# from ultralytics import YOLO

# # Load a YOLOv8n model
# model = YOLO("yolov8n.pt")  # You can choose a different model size like yolov8s.pt, yolov8m.pt, etc.

# # Train the model on the custom dataset
# # Assuming your dataset has a data.yaml file in the extracted directory
# data_config_path = './vehicle.v3i.yolov11/data.yaml'

# if os.path.exists(data_config_path):
#     results = model.train(data=data_config_path, epochs=100, imgsz=640) # You can adjust epochs and imgsz
#     print("Training finished.")
# else:
#     print(f"Error: data.yaml not found at {data_config_path}. Please ensure your dataset is correctly extracted and contains a data.yaml file.")

Ultralytics 8.3.177 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./vehicle.v3i.yolov11/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretra

Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

 22        [15, 18, 21]  1    752482  ultralytics.nn.modules.head.Detect           [6, [64, 128, 256]]           
Model summary: 129 layers, 3,012,018 parameters, 3,012,002 gradients, 8.2 GFLOPs

Transferred 319/355 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1333.4±741.0 MB/s, size: 65.3 KB)


train: Scanning /content/vehicle.v3i.yolov11/train/labels... 6590 images, 0 backgrounds, 0 corrupt: 100%|██████████| 6590/6590 [00:02<00:00, 2305.82it/s]


...

     16/100      3.04G      1.547      1.143      1.309        135        640: 100%|██████████| 412/412 [02:07<00:00,  3.23it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 26/26 [00:06<00:00,  3.74it/s]

                   all        823       4838      0.683      0.626      0.691      0.388



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     17/100      3.05G      1.542      1.117      1.298        161        640:  41%|████      | 168/412 [00:54<01:19,  3.08it/s]


KeyboardInterrupt: 

# Task
Analyze the video "pedestrian_and_vehicles.mp4" using the YOLOv8 model to find the minimum safe FPS that maintains the initial count of each detected object class. Start by reducing the FPS by 5 until the counts change, then increase the FPS by 2 until the counts return to the initial values.

## Load video and model

### Subtask:
Load the video file "pedestrian_and_vehicles.mp4" and a pre-trained YOLOv8 model.


**Reasoning**:
Import necessary libraries and load the video and the YOLO model.



In [19]:
# import cv2
# from ultralytics import YOLO

# # Define the path to the video file
# video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV"

# # Create a video capture object
# video_cap = cv2.VideoCapture(video_path)

# # Check if the video capture was successful
# if not video_cap.isOpened():
#     print(f"Error: Could not open video file {video_path}")

# # Load a pre-trained YOLOv8 model
# model = YOLO("yolov8n.pt")

**Reasoning**:
The video file was not found in the current directory. I need to check if the video file is available in the mounted Google Drive.



In [20]:
# # Define the path to the video file in Google Drive
# video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV"

# # Create a video capture object
# video_cap = cv2.VideoCapture(video_path)

# # Check if the video capture was successful
# if not video_cap.isOpened():
#     print(f"Error: Could not open video file {video_path}")
# else:
#     print(f"Successfully opened video file {video_path}")

# # The model is already loaded in the previous step, no need to load again.
# # model = YOLO("yolov8n.pt")

Successfully opened video file /content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV


**Reasoning**:
The video file was not found in the specified path in Google Drive. I need to check if the video file exists in a different location within Google Drive. I will try a common location for video files.



In [21]:
# # Define the path to the video file in Google Drive
# video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV"

# # Create a video capture object
# video_cap = cv2.VideoCapture(video_path)

# # Check if the video capture was successful
# if not video_cap.isOpened():
#     print(f"Error: Could not open video file {video_path}")
# else:
#     print(f"Successfully opened video file {video_path}")

# # The model is already loaded.

Successfully opened video file /content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV


**Reasoning**:
The video file was not found in the specified paths. I will try another common location for video files in Google Drive.



In [22]:
# # Define the path to the video file in Google Drive
# video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV"

# # Create a video capture object
# video_cap = cv2.VideoCapture(video_path)

# # Check if the video capture was successful
# if not video_cap.isOpened():
#     print(f"Error: Could not open video file {video_path}")
# else:
#     print(f"Successfully opened video file {video_path}")

# # The model is already loaded.

Successfully opened video file /content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV


**Reasoning**:
The video file was not found in the specified paths. I will try one more common location for video files in Google Drive. If the video is still not found, I will report failure as I cannot proceed with the subtask.



In [23]:
# # Define the path to the video file in Google Drive
# video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV"

# # Create a video capture object
# video_cap = cv2.VideoCapture(video_path)

# # Check if the video capture was successful
# if not video_cap.isOpened():
#     print(f"Error: Could not open video file {video_path}")
# else:
#     print(f"Successfully opened video file {video_path}")

# # The model is already loaded.

Successfully opened video file /content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV


## Initial object detection and counting

### Subtask:
Perform object detection on the original video to get the initial counts for each class.


**Reasoning**:
Perform object detection on a sample of frames to get the initial counts for each class, then print and reset the video capture.



In [24]:
# # Check if the video capture was successful
# if not video_cap.isOpened():
#     print(f"Error: Could not open video file {video_path}")
# else:
#     # Get the total number of frames
#     total_frames = int(video_cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     print(f"Total frames in the video: {total_frames}")

#     # Initialize a dictionary to store the counts for each detected class
#     initial_counts = {}

#     # Define the number of frames to sample
#     num_sampled_frames = 10 # Sample the first 10 frames as a representative sample

#     for i in range(num_sampled_frames):
#         ret, frame = video_cap.read()
#         if not ret:
#             print(f"Warning: Could not read frame {i}. Ending sampling.")
#             break

#         # Perform object detection
#         results = model(frame)

#         # Process the detection results and update counts
#         for r in results:
#             boxes = r.boxes
#             for box in boxes:
#                 class_id = int(box.cls[0])
#                 label = model.names[class_id]
#                 if label not in initial_counts:
#                     initial_counts[label] = 0
#                 initial_counts[label] += 1

#     # Print the initial counts
#     print("Initial object counts from sampled frames:")
#     print(initial_counts)

#     # Reset the video capture to the beginning
#     video_cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
#     print("Video capture reset to the beginning.")

Total frames in the video: 4797

0: 480x640 1 person, 21 cars, 3 buss, 5 trucks, 47.5ms
Speed: 10.4ms preprocess, 47.5ms inference, 313.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 21 cars, 3 buss, 5 trucks, 7.1ms
Speed: 3.4ms preprocess, 7.1ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 22 cars, 3 buss, 4 trucks, 6.5ms
Speed: 2.4ms preprocess, 6.5ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 21 cars, 2 buss, 4 trucks, 6.4ms
Speed: 1.6ms preprocess, 6.4ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 22 cars, 2 buss, 4 trucks, 6.4ms
Speed: 1.9ms preprocess, 6.4ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 24 cars, 3 buss, 4 trucks, 7.2ms
Speed: 1.7ms preprocess, 7.2ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 22 cars, 3 buss, 4 trucks, 6.4ms
S

In [40]:
# import cv2
# from ultralytics import YOLO
# import os

# def count_objects_in_segment(video_path, model, start_frame, end_frame, frame_skip_interval=1):
#     """
#     Counts objects in a specified video segment with a given frame skip interval.

#     Args:
#         video_path (str): Path to the video file.
#         model (YOLO): Loaded YOLO model.
#         start_frame (int): The starting frame of the segment (inclusive).
#         end_frame (int): The ending frame of the segment (exclusive).
#         frame_skip_interval (int): The number of frames to skip between processing.

#     Returns:
#         dict: A dictionary containing the total counts of each detected object class
#               within the processed frames of the segment.
#     """
#     video_cap = cv2.VideoCapture(video_path)
#     if not video_cap.isOpened():
#         print(f"Error: Could not open video file {video_path}")
#         return None

#     video_cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

#     current_counts = {}
#     processed_frames_count = 0

#     for i in range(start_frame, end_frame, frame_skip_interval):
#         video_cap.set(cv2.CAP_PROP_POS_FRAMES, i)
#         ret, frame = video_cap.read()
#         if not ret:
#             # If we can't read the frame we are supposed to process after seeking,
#             # it might mean we are at the end or there's an issue.
#             # However, the loop range should handle the end correctly.
#             # This break is mainly for unexpected read errors within the range.
#             break

#         processed_frames_count += 1

#         # Perform object detection
#         results = model(frame)

#         # Process the detection results and update counts
#         for r in results:
#             boxes = r.boxes
#             for box in boxes:
#                 class_id = int(box.cls[0])
#                 label = model.names[class_id]
#                 if label not in current_counts:
#                     current_counts[label] = 0
#                 current_counts[label] += 1

#     video_cap.release()
#     return current_counts

# # --- Main Analysis ---

# # Define the path to your video file
# video_path = "/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part2.MOV" # Change this to your video path

# # Load a pre-trained YOLOv8 model
# # Make sure you have run the cell to install ultralytics and opencv-python
# model = YOLO("yolov8n.pt")

# # Check if the video file exists
# if not os.path.exists(video_path):
#     print(f"Error: Video file not found at {video_path}")
# else:
#     # Get video properties to define the segment
#     video_cap_check = cv2.VideoCapture(video_path)
#     if not video_cap_check.isOpened():
#          print(f"Error: Could not open video file {video_path} for initial check.")
#     else:
#         total_frames = int(video_cap_check.get(cv2.CAP_PROP_FRAME_COUNT))
#         video_cap_check.release() # Release the capture object after getting properties

#         # Define the start and end frame numbers for a representative segment
#         start_frame = 0
#         end_frame = min(200, total_frames) # Process the first 200 frames or fewer

#         print(f"Analyzing segment from frame {start_frame} to {end_frame-1}")

#         # 1. Initial Object Counting (Baseline)
#         print("\n--- Establishing Baseline Counts (Frame Skip = 1) ---")
#         baseline_counts = count_objects_in_segment(video_path, model, start_frame, end_frame, frame_skip_interval=1)

#         if baseline_counts is not None:
#             print("Baseline object counts:")
#             print(baseline_counts)

#             # 2. Iterative Frame Skipping
#             print("\n--- Starting Iterative Frame Skipping Analysis ---")
#             frame_skip_interval = 2 # Start with skipping 1 frame (process every 2nd frame)
#             maximum_safe_frame_skip = 1 # Initialize maximum safe skip to 1 (processing every frame)

#             # Define a tolerance for count comparison (e.g., 5% difference)
#             count_tolerance_percentage = 5

#             while frame_skip_interval < (end_frame - start_frame):
#                 print(f"\nTesting frame skip interval: {frame_skip_interval}")
#                 current_counts = count_objects_in_segment(video_path, model, start_frame, end_frame, frame_skip_interval)

#                 if current_counts is None:
#                     print("Error during object counting with frame skipping. Stopping analysis.")
#                     break

#                 print(f"Object counts at skip interval {frame_skip_interval}: {current_counts}")

#                 # Compare current counts with baseline counts
#                 counts_changed_significantly = False
#                 all_classes = set(baseline_counts.keys()).union(set(current_counts.keys()))

#                 for label in all_classes:
#                     baseline_count = baseline_counts.get(label, 0)
#                     current_count = current_counts.get(label, 0)

#                     # Avoid division by zero if baseline count is 0
#                     if baseline_count > 0:
#                         percentage_diff = abs(current_count - baseline_count) / baseline_count * 100
#                         if percentage_diff > count_tolerance_percentage:
#                             counts_changed_significantly = True
#                             print(f"  Significant change for class '{label}': Baseline={baseline_count}, Current={current_count}, Difference={percentage_diff:.2f}%")
#                             break # No need to check other classes if one changed significantly
#                     elif current_count > 0:
#                          # If baseline is 0 but current is > 0, it's a significant change
#                          counts_changed_significantly = True
#                          print(f"  Significant change for class '{label}': Baseline={baseline_count}, Current={current_count}")
#                          break


#                 if counts_changed_significantly:
#                     maximum_safe_frame_skip = frame_skip_interval - 1
#                     print(f"\nCounts changed significantly at frame skip interval {frame_skip_interval}.")
#                     print(f"Maximum safe frame skip interval: {maximum_safe_frame_skip}")
#                     break # Stop when counts change significantly
#                 else:
#                     print("  Counts are within tolerance. Increasing frame skip interval.")
#                     maximum_safe_frame_skip = frame_skip_interval # Update maximum safe skip
#                     frame_skip_interval += 1

#             # If the loop finishes without significant change, the last tested skip interval is the maximum safe one
#             if not counts_changed_significantly:
#                  print("\nFinished testing all skip intervals up to the segment length.")
#                  print(f"Maximum safe frame skip interval for this segment: {maximum_safe_frame_skip}")

#             # 3. Report Maximum Safe Frame Skip
#             print("\n--- Analysis Complete ---")
#             print(f"Determined maximum safe frame skip interval: {maximum_safe_frame_skip}")

#         else:
#             print("\n--- Analysis Failed ---")
#             print("Could not establish baseline counts. Please check the video path and file integrity.")

Analyzing segment from frame 0 to 199

--- Establishing Baseline Counts (Frame Skip = 1) ---

0: 480x640 1 person, 21 cars, 3 buss, 5 trucks, 17.2ms
Speed: 2.3ms preprocess, 17.2ms inference, 3.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 21 cars, 3 buss, 5 trucks, 8.8ms
Speed: 2.0ms preprocess, 8.8ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 22 cars, 3 buss, 4 trucks, 11.9ms
Speed: 2.1ms preprocess, 11.9ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 21 cars, 2 buss, 4 trucks, 12.8ms
Speed: 2.0ms preprocess, 12.8ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 22 cars, 2 buss, 4 trucks, 12.6ms
Speed: 2.0ms preprocess, 12.6ms inference, 2.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 24 cars, 3 buss, 4 trucks, 15.3ms
Speed: 2.0ms preprocess, 15.3ms inference, 2.6ms postprocess per image at shape (1, 3, 